# NYC Taxi Demand Forecasting: Data Collection

This notebook collects all the raw data for the project and puts it together into one dataset. Every trip gets the weather at its pickup zone for the hour it started in.

**Scope**

| Item | Value |
|---|---|
| Boroughs | Manhattan and Queens (138 TLC taxi zones) |
| Taxi types | Yellow and green |
| Period | 1 Sep 2025 to 28 Feb 2026 (local NYC time) |
| Taxi fields | pickup time, dropoff time, `PULocationID`, `DOLocationID` |
| Weather fields | apparent temperature, rain, snowfall, precipitation probability (hourly, per zone) |

**Sources**

* NYC TLC Trip Record Data: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
* Open-Meteo Historical Forecast API: https://open-meteo.com/en/docs/historical-forecast-api

**Steps**

1. Get the Manhattan and Queens taxi zones and work out a latitude/longitude for each one.
2. Call the Open-Meteo API for the hourly weather at every zone.
3. Download the monthly TLC trip files and keep the trips picked up in our zones.
4. Attach the weather to every trip, using its pickup zone and pickup hour.
5. Print a profile of the final dataset (these are the numbers used in `docs/`).

**How to run**

Run all cells from top to bottom. The first run downloads about 425 MB of TLC files and takes around 10 minutes. The files are cached in `data/raw/`, so later runs skip the downloads. The weather step always calls the API again and takes about 4 minutes because of the pauses between requests.

Packages needed: `pip install -r requirements.txt` (pandas, pyarrow, requests, pyshp, pyproj).

Full documentation of the method and every field: `docs/data_collection.md` and `docs/data_dictionary.md`.

## 0. Setup

### 0.1 Imports

* `pandas` and `pyarrow` read and write the Parquet files that TLC publishes.
* `requests` handles both the file downloads and the Open-Meteo API calls.
* `pyshp` (imported as `shapefile`) reads the TLC taxi zone boundaries, and `pyproj` converts their coordinates to latitude/longitude.

In [1]:
import logging
import sys
import time
import zipfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import requests
import shapefile  # the pyshp package
from pyproj import CRS, Transformer

### 0.2 Settings

All the settings for the collection are in this one cell: the folders, the scope (boroughs, taxi types, months), the source URLs and the weather variables. To change the scope, for example to add a borough or a month, edit this cell and rerun the notebook.

Things to note:

* **Folders.** The notebook sits in `scripts/`, so the project root is one level up. `data/raw/` holds the downloads and `data/interim/` holds the large trip files. Both are ignored by git. `data/processed/` and `data/external/` hold the small outputs that are committed.
* **Timestamps.** TLC timestamps are local New York time with no timezone attached. We request the weather in the same timezone (`America/New_York`) so the hours line up.
* **Column names.** Yellow and green files name the timestamp columns differently (`tpep_` for yellow, `lpep_` for green), so we keep a small mapping and rename both to `pickup_datetime` / `dropoff_datetime`.
* **Weather timing.** Open-Meteo gives `apparent_temperature` as the value at the timestamp. It gives `rain`, `snowfall` and `precipitation_probability` for the hour *before* the timestamp, so the 07:00 value covers 06:00 to 07:00. Step 2 corrects for this.

In [2]:
# Folders (the notebook lives in scripts/, the data folder is at the project root)
ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"

# Scope of the collection
BOROUGHS = ["Manhattan", "Queens"]
TAXI_TYPES = ["yellow", "green"]
MONTHS = ["2025-09", "2025-10", "2025-11", "2025-12", "2026-01", "2026-02"]
START_DATE = "2025-09-01"  # inclusive
END_DATE = "2026-02-28"    # inclusive
TIMEZONE = "America/New_York"  # TLC timestamps are local NYC time

# NYC TLC trip record data
TLC_BASE_URL = "https://d37ci6vzurychx.cloudfront.net"
TRIP_URL = TLC_BASE_URL + "/trip-data/{taxi_type}_tripdata_{month}.parquet"
ZONE_LOOKUP_URL = TLC_BASE_URL + "/misc/taxi_zone_lookup.csv"
ZONE_SHAPES_URL = TLC_BASE_URL + "/misc/taxi_zones.zip"

# Yellow files use a "tpep_" prefix and green files "lpep_" for the timestamps
TIMESTAMP_COLUMNS = {
    "yellow": ("tpep_pickup_datetime", "tpep_dropoff_datetime"),
    "green": ("lpep_pickup_datetime", "lpep_dropoff_datetime"),
}

# Open-Meteo Historical Forecast API
WEATHER_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"
WEATHER_VARIABLES = ["apparent_temperature", "rain", "snowfall", "precipitation_probability"]
# These three are reported for the hour before the timestamp (07:00 = 06:00-07:00)
PRECEDING_HOUR_VARIABLES = ["rain", "snowfall", "precipitation_probability"]

# Output files
ZONES_FILE = EXTERNAL_DIR / "taxi_zones_manhattan_queens.csv"
WEATHER_FILE = PROCESSED_DIR / "weather_hourly_by_zone.parquet"
TRIPS_DIR = INTERIM_DIR / "trips"                         # one file per taxi type and month
TRIPS_WITH_WEATHER_DIR = INTERIM_DIR / "trips_with_weather"  # same files with weather added
TAXI_SUMMARY_FILE = PROCESSED_DIR / "taxi_collection_summary.csv"

for folder in (RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)

Project root: D:\College\nyc-taxi-demand-forecasting


### 0.3 Helper functions

These helpers are used by every step:

* **`log`** prints timestamped progress messages, so it is clear where a long run has got to.
* **`get_with_retry`** makes an HTTP GET request and tries again if it fails. Downloads and API calls sometimes fail for temporary reasons: a network hiccup, a DNS error, or the server returning a 5xx error. HTTP **429** means we hit the Open-Meteo rate limit, and the limit resets every minute, so in that case we wait a full minute before trying again. Other 4xx errors (for example a wrong parameter) are not retried, because retrying would not fix them.
* **`download_file`** streams a file to disk in 1 MB chunks, so a 75 MB file never has to sit in memory. It writes to a temporary `.part` file first and renames it at the end, so an interrupted download never leaves a broken file behind. If the file is already there, it is not downloaded again.

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger("collection")


def get_with_retry(url, params=None, retries=5, timeout=120, stream=False):
    # GET a URL, retrying on network errors and on 429/5xx responses
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout, stream=stream)
            if response.status_code == 429 or response.status_code >= 500:
                raise requests.HTTPError(f"HTTP {response.status_code}", response=response)
            response.raise_for_status()
            return response
        except (requests.ConnectionError, requests.Timeout, requests.HTTPError) as err:
            status = getattr(err.response, "status_code", None)
            # give up on the last attempt, or on client errors that a retry won't fix
            if attempt == retries or (status is not None and status < 500 and status != 429):
                raise
            # 429 = rate limit, which resets every minute
            wait = 65 if status == 429 else 5 * attempt
            log.warning("Request failed (%s), retrying in %ss", err, wait)
            time.sleep(wait)


def download_file(url, dest, force=False):
    # download url to dest, skipping files we already have
    dest = Path(dest)
    if dest.exists() and not force:
        log.info("Already downloaded: %s", dest.name)
        return dest

    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    log.info("Downloading %s", url)
    with get_with_retry(url, stream=True, timeout=300) as response:
        with open(tmp, "wb") as f:
            for chunk in response.iter_content(chunk_size=1 << 20):  # 1 MB at a time
                f.write(chunk)
    tmp.replace(dest)  # only now does the real file name appear
    log.info("Saved %s (%.1f MB)", dest.name, dest.stat().st_size / 1e6)
    return dest

## Step 1: Taxi zones and their coordinates

The trip records do not have coordinates. They only give a pickup and dropoff **zone ID** (`PULocationID`, `DOLocationID`) from 1 to 265. The weather API needs a latitude and longitude, so we need a point for every zone.

TLC publishes two files for this:

* `taxi_zone_lookup.csv`: zone ID → zone name → borough. We use it to find the zones in Manhattan and Queens.
* `taxi_zones.zip`: a shapefile with the boundary polygon of every zone. The coordinates are in the New York State Plane system (EPSG:2263), measured in feet, not in degrees.

For each zone we take the **centroid** (centre of mass) of its polygon, then convert that point from State Plane feet to WGS84 latitude/longitude with `pyproj`.

### 1.1 Zone lookup

Download the lookup table and keep only the rows for our boroughs. If a borough name is misspelled in the settings, the function stops with an error instead of quietly returning fewer zones.

In [4]:
def load_zone_lookup():
    path = download_file(ZONE_LOOKUP_URL, RAW_DIR / "taxi_zone_lookup.csv")
    return pd.read_csv(path)


def borough_zones(lookup, boroughs=BOROUGHS):
    zones = lookup[lookup["Borough"].isin(boroughs)].copy()
    missing = set(boroughs) - set(zones["Borough"])
    if missing:
        raise ValueError(f"No zones found for boroughs {sorted(missing)}")
    return zones.sort_values("LocationID").reset_index(drop=True)

### 1.2 Zone centroids

The centroid of a polygon is found with the **shoelace formula**. It walks around the outline, and each edge adds a small signed area and its contribution to the centre point.

Some zones are made of several pieces, such as islands, and some have holes. In a shapefile, outer rings and holes are drawn in opposite directions, so their signed areas have opposite signs. Adding up all the rings therefore subtracts the holes automatically, and the result is the true area-weighted centre of the whole zone.

The calculation is done in State Plane coordinates. They are in feet on a flat grid, so the formula is exact there. Only the final point is converted to latitude/longitude.

In [5]:
def ring_area_and_centroid(points):
    # signed area and centroid of one closed ring (shoelace formula)
    area = cx = cy = 0.0
    for (x0, y0), (x1, y1) in zip(points, points[1:] + points[:1]):
        cross = x0 * y1 - x1 * y0
        area += cross
        cx += (x0 + x1) * cross
        cy += (y0 + y1) * cross
    area /= 2
    if area == 0:
        return 0.0, 0.0, 0.0
    return area, cx / (6 * area), cy / (6 * area)


def shape_centroid(shape):
    # area-weighted centroid over all rings; holes have negative area so they subtract
    starts = list(shape.parts) + [len(shape.points)]
    total_area = sum_x = sum_y = 0.0
    for start, end in zip(starts, starts[1:]):
        area, cx, cy = ring_area_and_centroid(shape.points[start:end])
        total_area += area
        sum_x += cx * area
        sum_y += cy * area
    return sum_x / total_area, sum_y / total_area


def zone_centroids():
    # LocationID, latitude, longitude for every zone in the TLC shapefile
    zip_path = download_file(ZONE_SHAPES_URL, RAW_DIR / "taxi_zones.zip")
    shp_dir = RAW_DIR / "taxi_zones"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW_DIR)

    # the .prj file says which coordinate system the shapes use (EPSG:2263)
    source_crs = CRS.from_wkt((shp_dir / "taxi_zones.prj").read_text())
    to_wgs84 = Transformer.from_crs(source_crs, "EPSG:4326", always_xy=True)

    rows = []
    with shapefile.Reader(str(shp_dir / "taxi_zones.shp")) as reader:
        for record in reader.iterShapeRecords():
            x, y = shape_centroid(record.shape)
            lon, lat = to_wgs84.transform(x, y)  # always_xy: input and output are (x, y) = (lon, lat)
            rows.append({
                "LocationID": int(record.record["LocationID"]),
                "latitude": round(lat, 5),   # 5 decimals is about 1 m
                "longitude": round(lon, 5),
            })
    return pd.DataFrame(rows)

### 1.3 Run Step 1

Join the centroids onto the lookup rows by `LocationID`. `validate="one_to_one"` makes pandas raise an error if a zone ID appears twice on either side. If any of our zones has no shape, we stop instead of carrying a zone with no coordinates into the weather step.

The result is saved to `data/external/taxi_zones_manhattan_queens.csv`. As a quick check, compare a few well-known zones with a map: Central Park (43), JFK Airport (132) and LaGuardia Airport (138).

In [6]:
zones = borough_zones(load_zone_lookup())
zones = zones.merge(zone_centroids(), on="LocationID", how="left", validate="one_to_one")

missing = zones[zones["latitude"].isna()]
if not missing.empty:
    raise ValueError(f"No shape found for zones: {missing['LocationID'].tolist()}")

zones.to_csv(ZONES_FILE, index=False)
log.info("%d zones %s saved to %s", len(zones), zones["Borough"].value_counts().to_dict(), ZONES_FILE.name)

zones[zones["LocationID"].isin([43, 132, 138])]

11:00:16 INFO    Already downloaded: taxi_zone_lookup.csv


11:00:16 INFO    Already downloaded: taxi_zones.zip


11:00:16 INFO    138 zones {'Queens': 69, 'Manhattan': 69} saved to taxi_zones_manhattan_queens.csv


,LocationID,Borough,Zone,service_zone,latitude,longitude
18,43,Manhattan,Central Park,Yellow Zone,40.78248,-73.96556
64,132,Queens,JFK Airport,Airports,40.64699,-73.78653
68,138,Queens,LaGuardia Airport,Airports,40.77438,-73.87363


## Step 2: Hourly weather for every zone (Open-Meteo API)

For every zone centroid from Step 1, we ask the Open-Meteo **Historical Forecast API** for the four hourly variables over the whole study period.

Request parameters:

| Parameter | Value | Reason |
|---|---|---|
| `latitude`, `longitude` | zone centroids, comma-separated | one result per zone |
| `start_date`, `end_date` | `2025-09-01` to `2026-03-01` | study period plus one extra day (explained in 2.2) |
| `hourly` | `apparent_temperature,rain,snowfall,precipitation_probability` | the four variables we need |
| `timezone` | `America/New_York` | hours come back in NYC local time, the same as the TLC timestamps |

The API accepts several coordinates in one request and returns a list with one result per coordinate, in the same order. We send **10 zones per request**, so 138 zones need 14 requests.

**Rate limit.** The free API allows 600 "calls" per minute. Open-Meteo counts a request by its size: one location over six months is about 13 calls, so a 10-zone request is about 130. With a 15-second pause between requests we stay under the limit. If we still hit it, `get_with_retry` waits a minute and tries again.

Each result also includes the latitude, longitude and elevation of the model **grid cell** Open-Meteo actually used. We keep those, so it is possible to see which zones share a grid cell.

In [7]:
LOCATIONS_PER_REQUEST = 10
PAUSE_BETWEEN_REQUESTS = 15  # seconds, keeps us under 600 calls per minute


def fetch_weather(zone_batch):
    # hourly weather for a batch of zones, returned as one DataFrame per zone
    params = {
        "latitude": ",".join(zone_batch["latitude"].astype(str)),
        "longitude": ",".join(zone_batch["longitude"].astype(str)),
        "start_date": START_DATE,
        # one extra day so the last hour can be filled after the shift in align_to_hour_start
        "end_date": str((pd.Timestamp(END_DATE) + pd.Timedelta(days=1)).date()),
        "hourly": ",".join(WEATHER_VARIABLES),
        "timezone": TIMEZONE,
    }
    results = get_with_retry(WEATHER_URL, params=params).json()
    if isinstance(results, dict):  # a single location comes back as an object, not a list
        results = [results]
    if len(results) != len(zone_batch):
        raise ValueError(f"Expected {len(zone_batch)} locations, got {len(results)}")

    frames = []
    for zone_id, result in zip(zone_batch["LocationID"], results):
        hourly = pd.DataFrame(result["hourly"])
        hourly.insert(0, "LocationID", int(zone_id))
        # the grid cell the API actually used for this point
        hourly["grid_latitude"] = result["latitude"]
        hourly["grid_longitude"] = result["longitude"]
        hourly["grid_elevation_m"] = result["elevation"]
        frames.append(hourly)
    return frames

### 2.1 Checks on the weather

**Aligning the hours.** A weather row should describe the hour that starts at its timestamp, because Step 4 matches a trip at 07:42 to the 07:00 row. Open-Meteo reports `rain`, `snowfall` and `precipitation_probability` for the **preceding** hour, so the 08:00 value is really the rain between 07:00 and 08:00. We shift these three columns back one row within each zone. After the shift, the 07:00 row holds the temperature at 07:00 and the rain, snow and probability for 07:00 to 08:00. The last hour of the period (28 Feb 23:00) needs the 1 Mar 00:00 value, which is why the request asks for one extra day. That day is then trimmed off.

**Completeness.** Every zone must have exactly one row for every hour of the period: 181 days × 24 = 4,344 hours. Missing zones, missing hours or duplicate hours stop the notebook with an error. Missing weather values are reported as a warning.

*Daylight saving:* on 2 Nov 2025 the clocks went back, so 01:00 to 01:59 happened twice. Open-Meteo returns a single 01:00 row for that day, which matches TLC, where both hours also look like 01:xx.

In [8]:
def align_to_hour_start(weather):
    # shift the preceding-hour variables so each row covers [hour, hour + 1)
    weather = weather.sort_values(["LocationID", "weather_hour"]).reset_index(drop=True)
    cols = PRECEDING_HOUR_VARIABLES
    weather[cols] = weather.groupby("LocationID")[cols].shift(-1)
    # the shift turns the column into float, so make the probability a whole number again
    weather["precipitation_probability"] = weather["precipitation_probability"].round().astype("Int64")
    last_hour = pd.Timestamp(f"{END_DATE} 23:00")
    return weather[weather["weather_hour"] <= last_hour].reset_index(drop=True)


def check_weather(weather, zones):
    # every zone needs exactly one row for every hour of the study period
    expected_hours = pd.date_range(START_DATE, f"{END_DATE} 23:00", freq="h")
    per_zone = weather.groupby("LocationID")["weather_hour"].agg(["count", "nunique"])
    if set(per_zone.index) != set(zones["LocationID"]):
        raise ValueError("Weather is missing for some zones")
    if (per_zone["count"] != len(expected_hours)).any() or (per_zone["nunique"] != len(expected_hours)).any():
        raise ValueError(f"Expected {len(expected_hours)} unique hours per zone:\n{per_zone}")

    nulls = weather[WEATHER_VARIABLES].isna().sum()
    if nulls.any():
        log.warning("Missing weather values:\n%s", nulls[nulls > 0].to_string())
    log.info("Weather check passed: %d zones x %d hours", len(per_zone), len(expected_hours))

### 2.2 Run Step 2

Send the zones in batches of 10 with a pause between requests, stack the results, convert the time strings to real timestamps, align the hours, check the table and save it to `data/processed/weather_hourly_by_zone.parquet`.

This table has weather for **every** zone and hour (138 × 4,344 = 599,472 rows), including hours when no taxi was picked up.

In [9]:
zones = pd.read_csv(ZONES_FILE)

frames = []
for start in range(0, len(zones), LOCATIONS_PER_REQUEST):
    if start:
        time.sleep(PAUSE_BETWEEN_REQUESTS)
    batch = zones.iloc[start:start + LOCATIONS_PER_REQUEST]
    log.info("Requesting weather for zones %s", batch["LocationID"].tolist())
    frames.extend(fetch_weather(batch))

weather = pd.concat(frames, ignore_index=True)
weather = weather.rename(columns={"time": "weather_hour"})
weather["weather_hour"] = pd.to_datetime(weather["weather_hour"])
weather = align_to_hour_start(weather)
check_weather(weather, zones)

weather.to_parquet(WEATHER_FILE, index=False)
log.info("Saved %d hourly rows for %d zones to %s",
         len(weather), weather["LocationID"].nunique(), WEATHER_FILE.name)

weather.head()

11:00:16 INFO    Requesting weather for zones [2, 4, 7, 8, 9, 10, 12, 13, 15, 16]


11:00:35 INFO    Requesting weather for zones [19, 24, 27, 28, 30, 38, 41, 42, 43, 45]


11:00:53 INFO    Requesting weather for zones [48, 50, 53, 56, 57, 64, 68, 70, 73, 74]


11:01:10 INFO    Requesting weather for zones [75, 79, 82, 83, 86, 87, 88, 90, 92, 93]


11:01:33 INFO    Requesting weather for zones [95, 96, 98, 100, 101, 102, 103, 104, 105, 107]


11:01:50 INFO    Requesting weather for zones [113, 114, 116, 117, 120, 121, 122, 124, 125, 127]


... (7 outputs left out)

11:04:07 INFO    Requesting weather for zones [249, 252, 253, 258, 260, 261, 262, 263]


11:04:10 INFO    Weather check passed: 138 zones x 4344 hours


11:04:11 INFO    Saved 599472 hourly rows for 138 zones to weather_hourly_by_zone.parquet


,LocationID,weather_hour,apparent_temperature,rain,snowfall,precipitation_probability,grid_latitude,grid_longitude,grid_elevation_m
0,2,2025-09-01 00:00:00,17.2,0.0,0.0,0,40.617157,-73.80485,-1.0
1,2,2025-09-01 01:00:00,16.6,0.0,0.0,0,40.617157,-73.80485,-1.0
2,2,2025-09-01 02:00:00,15.8,0.0,0.0,0,40.617157,-73.80485,-1.0
3,2,2025-09-01 03:00:00,16.0,0.0,0.0,0,40.617157,-73.80485,-1.0
4,2,2025-09-01 04:00:00,14.9,0.0,0.0,0,40.617157,-73.80485,-1.0


## Step 3: Taxi trips (TLC trip records)

TLC publishes one Parquet file per taxi type per month at a fixed URL. We need 6 months × 2 taxi types = **12 files**. Each file covers the **whole city**: a yellow month has around 3.5 to 4.4 million trips.

For each file:

1. **Download** it into `data/raw/trips/`, or skip the download if it is already there.
2. **Read only what we need.** We read just the four columns from the instructions (pickup time, dropoff time, `PULocationID`, `DOLocationID`) and pass a filter `PULocationID in <our 138 zones>` to the Parquet reader. The rest of the city's trips and the other 16 columns never get loaded into memory.
3. **Rename** the `tpep_`/`lpep_` timestamp columns to `pickup_datetime` / `dropoff_datetime` and add a `taxi_type` column. After this, yellow and green trips have identical columns and can be stacked.
4. **Keep only the study period.** A few trips in every TLC file carry wrong dates from meter clock errors (other months, sometimes other years). There is no weather for those, so we drop them and count how many.
5. **Save** the result as `data/interim/trips/{taxi_type}_{month}.parquet`.

The files are processed **one at a time**. All 23 million trips together would need several GB of memory.

A trip counts as ours when it was **picked up** in Manhattan or Queens. The drop-off can be anywhere.

In [10]:
UNKNOWN_ZONES = [264, 265]  # TLC "Unknown" and "Outside of NYC"


def download_month(taxi_type, month):
    url = TRIP_URL.format(taxi_type=taxi_type, month=month)
    return download_file(url, RAW_DIR / "trips" / f"{taxi_type}_tripdata_{month}.parquet")


def read_borough_trips(path, taxi_type, zone_ids):
    # the four columns we need, only for trips picked up in our zones
    pickup_col, dropoff_col = TIMESTAMP_COLUMNS[taxi_type]
    trips = pd.read_parquet(
        path,
        columns=[pickup_col, dropoff_col, "PULocationID", "DOLocationID"],
        filters=[("PULocationID", "in", zone_ids)],  # applied while reading the file
    )
    trips = trips.rename(columns={pickup_col: "pickup_datetime", dropoff_col: "dropoff_datetime"})
    trips.insert(0, "taxi_type", taxi_type)
    return trips


def in_study_period(trips):
    # True for trips whose pickup time is inside the study period
    start = pd.Timestamp(START_DATE)
    end = pd.Timestamp(END_DATE) + pd.Timedelta(days=1)
    return trips["pickup_datetime"].ge(start) & trips["pickup_datetime"].lt(end)

### 3.1 Data-quality counts

Removing bad records is part of the **data preparation** stage, not collection, so nothing is removed here. The collection step does count the suspicious records in each file, so the next stage knows what to expect:

* **duplicate_rows**: trips with exactly the same taxi type, pickup time, dropoff time, pickup zone and dropoff zone as another trip in the same file
* **dropoff_not_after_pickup**: the dropoff time is at or before the pickup time
* **longer_than_6h**: very long trips, usually a meter that was left running
* **dropoff_zone_unknown**: the dropoff zone is 264 (Unknown) or 265 (Outside of NYC)

In [11]:
def quality_counts(trips):
    # suspicious records, counted but not removed (cleaning happens in data preparation)
    duration = trips["dropoff_datetime"] - trips["pickup_datetime"]
    return {
        "duplicate_rows": int(trips.duplicated().sum()),
        "dropoff_not_after_pickup": int((duration <= pd.Timedelta(0)).sum()),
        "longer_than_6h": int((duration > pd.Timedelta(hours=6)).sum()),
        "dropoff_zone_unknown": int(trips["DOLocationID"].isin(UNKNOWN_ZONES).sum()),
    }

### 3.2 Run Step 3

Loop over the 12 files. For each file we also add a row to an **audit table** that records where the file came from, its size, how many rows it had, how many we kept, and the quality counts. The table is saved as `data/processed/taxi_collection_summary.csv`, so every number in the documentation can be traced back to a specific file.

`taxi_type` is stored as a `category` column. It only has two values, and storing it this way saves a lot of memory and disk space over millions of rows.

In [12]:
zones = pd.read_csv(ZONES_FILE)
zone_ids = zones["LocationID"].astype(int).tolist()
borough_of = zones.set_index("LocationID")["Borough"]

TRIPS_DIR.mkdir(parents=True, exist_ok=True)
summary = []
for taxi_type in TAXI_TYPES:
    for month in MONTHS:
        path = download_month(taxi_type, month)
        trips = read_borough_trips(path, taxi_type, zone_ids)
        keep = in_study_period(trips)
        total_rows = pq.read_metadata(path).num_rows  # row count without loading the file
        log.info("%s %s: %d of %d trips start in %s (%d outside study period dropped)",
                 taxi_type, month, len(trips), total_rows, " or ".join(BOROUGHS), (~keep).sum())

        trips = trips[keep].sort_values("pickup_datetime").reset_index(drop=True)
        trips["taxi_type"] = trips["taxi_type"].astype("category")
        trips.to_parquet(TRIPS_DIR / f"{taxi_type}_{month}.parquet", index=False)

        by_borough = trips["PULocationID"].map(borough_of).value_counts()
        summary.append({
            "taxi_type": taxi_type,
            "month": month,
            "source_url": TRIP_URL.format(taxi_type=taxi_type, month=month),
            "file_size_mb": round(path.stat().st_size / 1e6, 1),
            "rows_in_file": total_rows,
            "borough_pickups": int(keep.size),
            "outside_period_dropped": int((~keep).sum()),
            "rows_kept": len(trips),
            **{f"{b.lower()}_pickups": int(by_borough.get(b, 0)) for b in BOROUGHS},
            **quality_counts(trips),
        })

summary = pd.DataFrame(summary)
summary.to_csv(TAXI_SUMMARY_FILE, index=False)
log.info("Kept %d trips in total. Summary saved to %s", summary["rows_kept"].sum(), TAXI_SUMMARY_FILE.name)

summary.drop(columns="source_url")

11:04:11 INFO    Already downloaded: yellow_tripdata_2025-09.parquet


11:04:11 INFO    yellow 2025-09: 4048172 of 4251015 trips start in Manhattan or Queens (6 outside study period dropped)


11:04:17 INFO    Already downloaded: yellow_tripdata_2025-10.parquet


11:04:18 INFO    yellow 2025-10: 4234338 of 4428699 trips start in Manhattan or Queens (0 outside study period dropped)


11:04:24 INFO    Already downloaded: yellow_tripdata_2025-11.parquet


11:04:25 INFO    yellow 2025-11: 3982596 of 4181444 trips start in Manhattan or Queens (3 outside study period dropped)


... (16 outputs left out)

11:04:48 INFO    Already downloaded: green_tripdata_2026-02.parquet


11:04:48 INFO    green 2026-02: 30502 of 37373 trips start in Manhattan or Queens (2 outside study period dropped)


11:04:48 INFO    Kept 23318979 trips in total. Summary saved to taxi_collection_summary.csv


,taxi_type,month,file_size_mb,rows_in_file,borough_pickups,outside_period_dropped,rows_kept,manhattan_pickups,queens_pickups,duplicate_rows,dropoff_not_after_pickup,longer_than_6h,dropoff_zone_unknown
0,yellow,2025-09,72.4,4251015,4048172,6,4048166,3633590,414576,69685,56954,1048,23462
1,yellow,2025-10,75.3,4428699,4234338,0,4234338,3818197,416141,60660,67701,1062,24555
2,yellow,2025-11,71.1,4181444,3982596,3,3982593,3607966,374627,43004,61767,1131,22633
3,yellow,2025-12,73.7,4305006,4096636,0,4096636,3707215,389421,43414,57821,1193,24230
4,yellow,2026-01,64.2,3724889,3525433,0,3525433,3180607,344826,35256,44911,941,18925
5,yellow,2026-02,58.7,3399866,3209714,4,3209710,2909331,300379,24060,40419,851,16518
6,green,2025-09,1.2,48893,40077,18,40059,27950,12109,108,24,77,536
7,green,2025-10,1.2,49416,40573,0,40573,29233,11340,107,16,132,484
8,green,2025-11,1.2,46912,38373,0,38373,27708,10665,115,23,116,406
9,green,2025-12,1.2,48236,39806,0,39806,28850,10956,141,19,154,468


## Step 4: Attach the weather to every trip

This step builds the final dataset. For every trip:

1. take its pickup zone (`PULocationID`)
2. look up that zone's borough, name and centroid latitude/longitude from Step 1 (the point the weather was fetched for)
3. round the pickup time **down** to the hour (`pickup_hour`), so a pickup at 07:42 gets `pickup_hour` 07:00
4. join the weather table from Step 2 on **(`PULocationID`, `pickup_hour`)**

Because of the alignment in Step 2, the trip gets the apparent temperature at 07:00 and the rain, snowfall and precipitation probability for 07:00 to 08:00, the hour the trip started in.

Both joins use `validate="many_to_one"`. Many trips can share a zone or a zone-hour, but each zone and each zone-hour must appear only once on the right-hand side. If they didn't, the join would silently duplicate trips. After the join we also check that the row count did not change and that no trip is missing weather.

Zone names and boroughs are stored as `category`, which keeps the files small.

In [13]:
OUTPUT_COLUMNS = [
    "taxi_type",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "pickup_borough",
    "pickup_zone",
    "pickup_zone_latitude",
    "pickup_zone_longitude",
    "pickup_hour",
    *WEATHER_VARIABLES,
]


def load_zones_for_join():
    zones = pd.read_csv(ZONES_FILE).rename(columns={
        "LocationID": "PULocationID",
        "Borough": "pickup_borough",
        "Zone": "pickup_zone",
        "latitude": "pickup_zone_latitude",
        "longitude": "pickup_zone_longitude",
    })
    zones = zones[["PULocationID", "pickup_borough", "pickup_zone",
                   "pickup_zone_latitude", "pickup_zone_longitude"]]
    for col in ("pickup_borough", "pickup_zone"):
        zones[col] = zones[col].astype("category")
    return zones


def load_weather_for_join():
    weather = pd.read_parquet(WEATHER_FILE)
    weather = weather.rename(columns={"LocationID": "PULocationID", "weather_hour": "pickup_hour"})
    # the trip timestamps are in microseconds, so the join key has to match
    weather["pickup_hour"] = weather["pickup_hour"].astype("datetime64[us]")
    return weather[["PULocationID", "pickup_hour", *WEATHER_VARIABLES]]


def add_weather(trips, zones, weather):
    data = trips.copy()
    data["pickup_hour"] = data["pickup_datetime"].dt.floor("h")
    data = data.merge(zones, on="PULocationID", how="left", validate="many_to_one")
    data = data.merge(weather, on=["PULocationID", "pickup_hour"], how="left", validate="many_to_one")
    return data[OUTPUT_COLUMNS]


def check_trips(data, trips, name):
    if len(data) != len(trips):
        raise ValueError(f"{name}: row count changed in the join ({len(trips)} -> {len(data)})")
    missing = data[WEATHER_VARIABLES + ["pickup_zone"]].isna().any(axis=1).sum()
    if missing:
        raise ValueError(f"{name}: {missing} trips did not get zone or weather data")

### 4.1 Run Step 4

As in Step 3, this goes one monthly file at a time. The output uses the same split: `data/interim/trips_with_weather/{taxi_type}_{month}.parquet`, 12 files with about 23.3 million rows in total (about 345 MB).

The files are too large for GitHub, so they stay out of git. Anyone can rebuild them by running this notebook. To load them later:

```python
trips = pd.read_parquet("data/interim/trips_with_weather/")                    # all 12 files (several GB of RAM)
nov = pd.read_parquet("data/interim/trips_with_weather/yellow_2025-11.parquet")  # one month
```

In [14]:
zones_for_join = load_zones_for_join()
weather_for_join = load_weather_for_join()

TRIPS_WITH_WEATHER_DIR.mkdir(parents=True, exist_ok=True)
total = 0
for taxi_type in TAXI_TYPES:
    for month in MONTHS:
        name = f"{taxi_type}_{month}"
        trips = pd.read_parquet(TRIPS_DIR / f"{name}.parquet")
        data = add_weather(trips, zones_for_join, weather_for_join)
        check_trips(data, trips, name)
        data.to_parquet(TRIPS_WITH_WEATHER_DIR / f"{name}.parquet", index=False)
        total += len(data)
        log.info("%s: %d trips, all with weather", name, len(data))

log.info("Saved %d trips with weather to %s", total, TRIPS_WITH_WEATHER_DIR)

11:04:57 INFO    yellow_2025-09: 4048166 trips, all with weather


11:05:06 INFO    yellow_2025-10: 4234338 trips, all with weather


11:05:13 INFO    yellow_2025-11: 3982593 trips, all with weather


11:05:20 INFO    yellow_2025-12: 4096636 trips, all with weather


11:05:27 INFO    yellow_2026-01: 3525433 trips, all with weather


11:05:30 INFO    yellow_2026-02: 3209710 trips, all with weather


... (3 outputs left out)

11:05:30 INFO    green_2025-12: 39806 trips, all with weather


11:05:30 INFO    green_2026-01: 32792 trips, all with weather


11:05:31 INFO    green_2026-02: 30500 trips, all with weather


11:05:31 INFO    Saved 23318979 trips with weather to D:\College\nyc-taxi-demand-forecasting\data\interim\trips_with_weather


A sample of the final dataset: the first few trips of November 2025 (yellow).

In [15]:
pd.read_parquet(TRIPS_WITH_WEATHER_DIR / "yellow_2025-11.parquet").head()

,taxi_type,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,pickup_borough,pickup_zone,pickup_zone_latitude,pickup_zone_longitude,pickup_hour,apparent_temperature,rain,snowfall,precipitation_probability
0,yellow,2025-10-31 23:36:10,2025-10-31 23:47:24,161,143,Manhattan,Midtown Center,40.75803,-73.97770,2025-10-31 23:00:00,5.0,0.0,0.0,0
1,yellow,2025-10-31 23:46:16,2025-10-31 23:56:13,263,239,Manhattan,Yorkville West,40.77877,-73.95101,2025-10-31 23:00:00,6.2,0.0,0.0,0
2,yellow,2025-10-31 23:49:20,2025-11-01 00:00:37,107,68,Manhattan,Gramercy,40.73682,-73.98405,2025-10-31 23:00:00,5.8,0.0,0.0,0
3,yellow,2025-10-31 23:50:14,2025-11-01 00:05:23,143,236,Manhattan,Lincoln Square West,40.77597,-73.98765,2025-10-31 23:00:00,5.4,0.0,0.0,0
4,yellow,2025-10-31 23:50:19,2025-10-31 23:57:03,163,163,Manhattan,Midtown North,40.76442,-73.97757,2025-10-31 23:00:00,5.2,0.0,0.0,0


## Step 5: Dataset profile

A short profile of what was collected. These are the numbers quoted in `docs/data_collection.md` and `docs/data_dictionary.md`.

The trip files are again read one at a time, keeping only running totals, so this also works on a machine with little memory.

### 5.1 From the raw files to the final dataset

In [16]:
summary = pd.read_csv(TAXI_SUMMARY_FILE)

print(f"rows in the 12 TLC files:          {summary['rows_in_file'].sum():,}")
print(f"picked up in Manhattan or Queens:  {summary['borough_pickups'].sum():,}")
print(f"outside study period (dropped):    {summary['outside_period_dropped'].sum():,}")
print(f"trips kept:                        {summary['rows_kept'].sum():,}")

rows in the 12 TLC files:          24,562,021
picked up in Manhattan or Queens:  23,319,012
outside study period (dropped):    33
trips kept:                        23,318,979


### 5.2 One pass over the final dataset

Collect all the totals in a single pass: missing values, trips by borough and month, trips per zone, trips by hour of day, where the trips ended, and how many started in rain or snow.

Dropoff zones can be anywhere, so they are mapped to boroughs with the full TLC lookup table. Zone 265 has the borough "N/A" in that table, which pandas reads as a missing value, so it is filled back in as "N/A".

In [17]:
PROFILE_COLUMNS = ["taxi_type", "pickup_datetime", "PULocationID", "DOLocationID",
                   "pickup_borough", "pickup_zone", *WEATHER_VARIABLES]
all_boroughs = pd.read_csv(RAW_DIR / "taxi_zone_lookup.csv").set_index("LocationID")["Borough"]


def add_counts(total, counts):
    # running total of value counts across the monthly files
    return counts if total is None else total.add(counts, fill_value=0)


rows = rain_trips = snow_trips = 0
missing = by_borough = by_month = by_zone = by_hour = dropoffs = None
first = last = None

for file in sorted(TRIPS_WITH_WEATHER_DIR.glob("*.parquet")):
    trips = pd.read_parquet(file, columns=PROFILE_COLUMNS)
    rows += len(trips)
    missing = add_counts(missing, trips.isna().sum())
    by_borough = add_counts(by_borough, trips.groupby(["pickup_borough", "taxi_type"], observed=True).size())
    month = trips["pickup_datetime"].dt.strftime("%Y-%m")
    by_month = add_counts(by_month, trips.groupby([month, "taxi_type"], observed=True).size())
    by_zone = add_counts(by_zone, trips["PULocationID"].value_counts())
    by_hour = add_counts(by_hour, trips["pickup_datetime"].dt.hour.value_counts())
    dropoffs = add_counts(dropoffs, trips["DOLocationID"].map(all_boroughs).fillna("N/A").value_counts())
    rain_trips += int((trips["rain"] > 0).sum())
    snow_trips += int((trips["snowfall"] > 0).sum())
    first = trips["pickup_datetime"].min() if first is None else min(first, trips["pickup_datetime"].min())
    last = trips["pickup_datetime"].max() if last is None else max(last, trips["pickup_datetime"].max())

print(f"rows: {rows:,}")
print(f"pickups from {first} to {last}")
print("\nmissing values per column:")
print(missing.astype("int64").to_string())

rows: 23,318,979
pickups from 2025-09-01 00:00:00 to 2026-02-28 23:59:59

missing values per column:
taxi_type                    0
pickup_datetime              0
PULocationID                 0
DOLocationID                 0
pickup_borough               0
pickup_zone                  0
apparent_temperature         0
rain                         0
snowfall                     0
precipitation_probability    0


### 5.3 Trips by borough and by month

Manhattan has far more yellow cab trips than Queens. Green cabs are a small share overall, but they matter in upper Manhattan and Queens, which is where they are allowed to pick up street hails.

Monthly totals here are by **pickup date**, so they differ very slightly from the per-file counts in the audit table. A file can include trips from the last minutes of the previous month.

In [18]:
print(by_borough.astype("int64").unstack(fill_value=0).to_string())
print()
print(by_month.astype("int64").unstack(fill_value=0).to_string())

taxi_type        green    yellow
pickup_borough                  
Manhattan       159610  20856906
Queens           62493   2239970

taxi_type        green   yellow
pickup_datetime                
2025-09          40052  4048175
2025-10          40575  4234347
2025-11          38375  3982584
2025-12          39799  4096633
2026-01          32792  3525437
2026-02          30510  3209700


### 5.4 Busiest and quietest zones

Zones with no pickups at all are added back with 0, so the list always has all 138 zones.

In [19]:
zone_names = pd.read_csv(ZONES_FILE).set_index("LocationID")[["Zone", "Borough"]]
zone_trips = by_zone.reindex(zone_names.index, fill_value=0).astype("int64").sort_values(ascending=False)
zone_table = zone_names.join(zone_trips.rename("trips")).loc[zone_trips.index]

print("Top 10 pickup zones")
print(zone_table.head(10).to_string())
print("\nZones with the fewest pickups")
print(zone_table.tail(5).to_string())

Top 10 pickup zones
                                    Zone    Borough    trips
LocationID                                                  
237                Upper East Side South  Manhattan  1090154
132                          JFK Airport     Queens   988812
161                       Midtown Center  Manhattan   988458
236                Upper East Side North  Manhattan   984375
186         Penn Station/Madison Sq West  Manhattan   748457
162                         Midtown East  Manhattan   733984
142                  Lincoln Square East  Manhattan   708020
230            Times Sq/Theatre District  Manhattan   699653
234                             Union Sq  Manhattan   635241
79                          East Village  Manhattan   627586

Zones with the fewest pickups
                                                     Zone    Borough  trips
LocationID                                                                 
27                    Breezy Point/Fort Tilden/Riis Beach     Que

### 5.5 Time of day, destinations and data quality

* Trips by the hour of the day they started in (summed over the whole period).
* The borough where the trips ended. "Unknown" and "N/A" are TLC zones 264 and 265.
* The quality counts from Step 3, added up over all files. These records are still in the dataset.

In [20]:
print("Trips by hour of day")
print(by_hour.astype("int64").sort_index().to_string())

print("\nDropoff borough")
print(dropoffs.astype("int64").sort_values(ascending=False).to_string())

print("\nData quality counts (for the data preparation step)")
quality = ["duplicate_rows", "dropoff_not_after_pickup", "longer_than_6h", "dropoff_zone_unknown"]
print(summary[quality].sum().to_string())

Trips by hour of day
pickup_datetime
0      711036
1      470160
2      310353
3      209994
4      161898
5      174021
6      353004
7      660602
8      882269
9      955215
10    1004704
11    1085966
12    1171173
13    1221517
14    1312612
15    1388607
16    1427641
17    1590410
18    1676922
19    1505106
20    1381886
21    1393751
22    1275872
23     994260

Dropoff borough
DOLocationID
Manhattan        20623472
Queens            1307421
Brooklyn           976952
Bronx              230637
N/A                102370
EWR                 41994
Unknown             30668
Staten Island        5465

Data quality counts (for the data preparation step)
duplicate_rows              276749
dropoff_not_after_pickup    329681
longer_than_6h                6877
dropoff_zone_unknown        133038


### 5.6 Weather

A summary of the weather table from Step 2, and how many trips started in an hour with rain or snow. The last line shows how many distinct Open-Meteo grid cells the 138 zone centroids fell into. Zones in the same grid cell get identical weather.

In [21]:
weather = pd.read_parquet(WEATHER_FILE)

print(f"trips in an hour with rain:     {rain_trips:,}")
print(f"trips in an hour with snowfall: {snow_trips:,}")
print()
print(weather[WEATHER_VARIABLES].describe().round(2).to_string())
print(f"\nzone-hours with rain > 0:     {(weather['rain'] > 0).mean():.1%}")
print(f"zone-hours with snowfall > 0: {(weather['snowfall'] > 0).mean():.1%}")

grid = weather.groupby("LocationID")[["grid_latitude", "grid_longitude"]].first()
print(f"distinct Open-Meteo grid cells: {len(grid.drop_duplicates())} for {weather['LocationID'].nunique()} zones")

trips in an hour with rain:     1,090,584
trips in an hour with snowfall: 312,615

       apparent_temperature       rain   snowfall  precipitation_probability
count             599472.00  599472.00  599472.00                   599472.0
mean                   3.48       0.07       0.02                       5.97
std                   11.70       0.79       0.13                      17.84
min                  -25.30       0.00       0.00                        0.0
25%                   -5.10       0.00       0.00                        0.0
50%                    1.60       0.00       0.00                        0.0
75%                   12.20       0.00       0.00                        1.0
max                   35.40      83.50       4.06                      100.0

zone-hours with rain > 0:     5.3%
zone-hours with snowfall > 0: 2.4%
distinct Open-Meteo grid cells: 50 for 138 zones


## Output files

| File | In git | Contents |
|---|---|---|
| `data/interim/trips_with_weather/*.parquet` | no (about 345 MB) | **Final dataset**: every trip with the weather at its pickup zone and hour |
| `data/processed/weather_hourly_by_zone.parquet` | yes | Hourly weather for every zone and hour |
| `data/processed/taxi_collection_summary.csv` | yes | Audit table, one row per TLC file |
| `data/external/taxi_zones_manhattan_queens.csv` | yes | The 138 zones with centroid latitude/longitude |
| `data/interim/trips/*.parquet` | no | Trips before the weather join |
| `data/raw/` | no | Downloaded TLC files |

Field descriptions, and the reasons for choosing these fields out of everything the two sources offer, are in `docs/data_dictionary.md`.